In [ ]:
# ============================================
# Rope dataset collection cell
# eval_object() pixel points -> LUT -> workspace
# grasp / pull / release / log to jsonl
# ============================================

from pathlib import Path
import json
import time
import math
import random
import numpy as np

from calibration.pixel_to_workspace import PixelToWorkspaceMapper


# -------------------------------------------------
# User config
# -------------------------------------------------
PROJECT_ROOT = Path("/path/to/RGMC")   # 예: Path("/Users/kimsumin/projects/RGMC")
ROBOT_ID = 2                           # 현재 연결된 로봇 번호에 맞게 수정
OUTPUT_JSONL = PROJECT_ROOT / "data" / "rope_dataset_robot.jsonl"

NUM_EPISODES = 20
STEPS_PER_EPISODE = 10

# Z / timing
Z_PICK = 0.0
Z_SAFE = 0.3
SETTLE_SEC_AFTER_RESET = 1.0
SETTLE_SEC_AFTER_RELEASE = 0.8
COMMAND_WAIT_SEC = 0.25

# Sampling
NODE_INDEX_RANGE = (1, 18)  # 끝점 불안정하면 0,19 제외하고 시작
DX_RANGE = (-0.05, 0.05)
DY_RANGE = (-0.05, 0.05)

# Workspace safety box
# LUT 자체는 pixel->workspace만 담당하므로,
# 여기서는 dataset 수집용 안전 영역만 따로 둠.
X_MIN, X_MAX = 0.08, 0.92
Y_MIN, Y_MAX = 0.08, 0.92

# Optional fixed rotation
ROTATE_BEFORE_PICK_DEG = None   # 예: 0 또는 None

# Random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


# -------------------------------------------------
# Mapper load
# -------------------------------------------------
LUT_PATH = PROJECT_ROOT / "data" / "map" / f"robot{ROBOT_ID}" / "lut.pkl"
if not LUT_PATH.exists():
    raise FileNotFoundError(f"LUT not found: {LUT_PATH}")

mapper = PixelToWorkspaceMapper(
    str(LUT_PATH),
    clamp_to_workspace=False,
)

print("LUT_PATH:", LUT_PATH)
print("OUTPUT_JSONL:", OUTPUT_JSONL)


# -------------------------------------------------
# Helpers
# -------------------------------------------------
def clip_xy(x, y, x_min=X_MIN, x_max=X_MAX, y_min=Y_MIN, y_max=Y_MAX):
    x = float(np.clip(x, x_min, x_max))
    y = float(np.clip(y, y_min, y_max))
    return x, y


def sleep_cmd(sec=COMMAND_WAIT_SEC):
    time.sleep(sec)


def extract_eval_object_points(eval_obj_result):
    """
    eval_object() 결과에서 pixel 20점 추출
    기대 형식:
      {
        "coordinate_space": "undistorted_pixel_2d",
        "geometry": {
            "type": "...",
            "points": [{"x": ..., "y": ...}, ...]
        }
      }
    """
    if eval_obj_result is None:
        raise RuntimeError("eval_object() returned None")

    geom = eval_obj_result.get("geometry", {})
    pts = geom.get("points", [])
    if len(pts) == 0:
        raise RuntimeError("No points found in eval_object().")

    uv = np.array([[float(p["x"]), float(p["y"])] for p in pts], dtype=np.float32)
    return uv


def pixel_points_to_workspace(mapper, uv_points):
    xy = np.array(mapper.convert_many([tuple(p) for p in uv_points]), dtype=np.float32)
    return xy


def get_rope_obs_workspace(robot, mapper):
    """
    eval_object() -> pixel 20점 -> workspace 20점
    """
    raw = robot.eval_object()
    uv = extract_eval_object_points(raw)
    xy = pixel_points_to_workspace(mapper, uv)

    return {
        "raw": raw,
        "uv_points": uv,   # (N, 2)
        "xy_points": xy,   # (N, 2)
    }


def sample_node_idx(num_nodes):
    lo, hi = NODE_INDEX_RANGE
    lo = max(0, lo)
    hi = min(num_nodes - 1, hi)
    return random.randint(lo, hi)


def sample_target_xy(grasp_xy):
    dx = random.uniform(*DX_RANGE)
    dy = random.uniform(*DY_RANGE)
    tx = grasp_xy[0] + dx
    ty = grasp_xy[1] + dy
    tx, ty = clip_xy(tx, ty)
    dx = tx - grasp_xy[0]
    dy = ty - grasp_xy[1]
    return np.array([tx, ty], dtype=np.float32), float(dx), float(dy)


def jsonable_points(arr):
    return [[float(x), float(y)] for x, y in np.asarray(arr)]


def append_jsonl(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def execute_rope_pick_and_pull(
    robot,
    grasp_xy,
    target_xy,
    z_pick=Z_PICK,
    z_safe=Z_SAFE,
    rotate_before_pick_deg=ROTATE_BEFORE_PICK_DEG,
):
    """
    open -> move_xy(grasp) -> z down -> close -> z up -> move_xy(target) -> open -> z safe
    """
    try:
        gx, gy = float(grasp_xy[0]), float(grasp_xy[1])
        tx, ty = float(target_xy[0]), float(target_xy[1])

        # 0) optional rotation
        if rotate_before_pick_deg is not None:
            robot.rotate(float(rotate_before_pick_deg))
            sleep_cmd()

        # 1) 안전 높이 + gripper open
        robot.move_z(float(z_safe))
        sleep_cmd()

        robot.gripper_open()
        sleep_cmd()

        # 2) grasp point 상공으로 이동
        robot.move_xy(gx, gy)
        sleep_cmd()

        # 3) 내려가서 집기
        robot.move_z(float(z_pick))
        sleep_cmd()

        robot.gripper_close()
        sleep_cmd()

        # 4) 들어올리기
        robot.move_z(float(z_safe))
        sleep_cmd()

        # 5) target으로 이동
        robot.move_xy(tx, ty)
        sleep_cmd()

        # 6) 놓기
        robot.gripper_open()
        sleep_cmd()

        # 7) rope에 안 걸리게 safe z 유지
        robot.move_z(float(z_safe))
        sleep_cmd()

        return True, None

    except Exception as e:
        return False, repr(e)


def estimate_motion_amount(before_xy, after_xy):
    before_xy = np.asarray(before_xy, dtype=np.float32)
    after_xy = np.asarray(after_xy, dtype=np.float32)
    if before_xy.shape != after_xy.shape:
        return None
    delta = after_xy - before_xy
    per_node = np.linalg.norm(delta, axis=1)
    return {
        "mean_node_disp": float(np.mean(per_node)),
        "max_node_disp": float(np.max(per_node)),
        "sum_node_disp": float(np.sum(per_node)),
    }


# -------------------------------------------------
# Main collection loop
# -------------------------------------------------
all_records = []

for episode_id in range(NUM_EPISODES):
    print("\n" + "=" * 90)
    print(f"[EPISODE {episode_id}] reset")
    print("=" * 90)

    reset_out = robot.env_reset()
    print("env_reset:", reset_out)
    time.sleep(SETTLE_SEC_AFTER_RESET)

    for step_id in range(STEPS_PER_EPISODE):
        print(f"\n[EP {episode_id:03d} | STEP {step_id:03d}]")

        # -----------------------------------------
        # BEFORE
        # -----------------------------------------
        try:
            obs_before = get_rope_obs_workspace(robot, mapper)
        except Exception as e:
            print("before observation failed:", e)
            rec = {
                "episode_id": episode_id,
                "step_id": step_id,
                "valid_sample": False,
                "stage": "before_observation",
                "error": repr(e),
            }
            append_jsonl(OUTPUT_JSONL, rec)
            continue

        before_uv = obs_before["uv_points"]
        before_xy = obs_before["xy_points"]

        num_nodes = len(before_xy)
        if num_nodes < 2:
            print("too few rope nodes:", num_nodes)
            rec = {
                "episode_id": episode_id,
                "step_id": step_id,
                "valid_sample": False,
                "stage": "before_observation",
                "error": f"too_few_nodes:{num_nodes}",
            }
            append_jsonl(OUTPUT_JSONL, rec)
            continue

        node_idx = sample_node_idx(num_nodes)
        grasp_xy = before_xy[node_idx].copy()
        target_xy, dx, dy = sample_target_xy(grasp_xy)

        print("num_nodes :", num_nodes)
        print("node_idx  :", node_idx)
        print("grasp_xy  :", grasp_xy)
        print("target_xy :", target_xy)
        print("dx, dy    :", dx, dy)

        # -----------------------------------------
        # EXECUTE
        # -----------------------------------------
        execute_ok, execute_error = execute_rope_pick_and_pull(
            robot=robot,
            grasp_xy=grasp_xy,
            target_xy=target_xy,
            z_pick=Z_PICK,
            z_safe=Z_SAFE,
            rotate_before_pick_deg=ROTATE_BEFORE_PICK_DEG,
        )

        print("execute_ok:", execute_ok, "| execute_error:", execute_error)

        time.sleep(SETTLE_SEC_AFTER_RELEASE)

        # -----------------------------------------
        # AFTER
        # -----------------------------------------
        try:
            obs_after = get_rope_obs_workspace(robot, mapper)
            after_uv = obs_after["uv_points"]
            after_xy = obs_after["xy_points"]
            obs_error = None
        except Exception as e:
            after_uv = None
            after_xy = None
            obs_error = repr(e)
            print("after observation failed:", obs_error)

        motion_stats = None
        grasp_success_like = None
        if after_xy is not None and execute_ok:
            motion_stats = estimate_motion_amount(before_xy, after_xy)
            # 매우 단순한 휴리스틱: rope 전체 평균 변화가 너무 작으면 grasp 실패 후보
            grasp_success_like = motion_stats["mean_node_disp"] > 0.002

        # -----------------------------------------
        # LOG
        # -----------------------------------------
        record = {
            "episode_id": episode_id,
            "step_id": step_id,
            "timestamp": time.time(),

            "valid_sample": bool(execute_ok and after_xy is not None),
            "execute_ok": bool(execute_ok),
            "execute_error": execute_error,
            "after_observation_error": obs_error,

            "node_idx": int(node_idx),

            "grasp_xy": [float(grasp_xy[0]), float(grasp_xy[1])],
            "target_xy": [float(target_xy[0]), float(target_xy[1])],
            "dx": float(dx),
            "dy": float(dy),

            "z_pick": float(Z_PICK),
            "z_safe": float(Z_SAFE),
            "rotate_before_pick_deg": None if ROTATE_BEFORE_PICK_DEG is None else float(ROTATE_BEFORE_PICK_DEG),

            "state_before_uv": jsonable_points(before_uv),
            "state_before_xy": jsonable_points(before_xy),

            "state_after_uv": None if after_uv is None else jsonable_points(after_uv),
            "state_after_xy": None if after_xy is None else jsonable_points(after_xy),

            "motion_stats": motion_stats,
            "grasp_success_like": grasp_success_like,
        }

        append_jsonl(OUTPUT_JSONL, record)
        all_records.append(record)

        print("saved ->", OUTPUT_JSONL)
        if motion_stats is not None:
            print("motion_stats:", motion_stats)

print("\nDone.")
print("total records:", len(all_records))
print("jsonl path:", OUTPUT_JSONL)

In [ ]:
# 마지막 몇 개 샘플 확인
from pathlib import Path
import json

path = Path(OUTPUT_JSONL)
lines = path.read_text(encoding="utf-8").strip().splitlines()

for line in lines[-3:]:
    rec = json.loads(line)
    print("=" * 80)
    print("episode_id:", rec["episode_id"], "step_id:", rec["step_id"])
    print("node_idx  :", rec["node_idx"])
    print("grasp_xy  :", rec["grasp_xy"])
    print("target_xy :", rec["target_xy"])
    print("execute_ok:", rec["execute_ok"])
    print("valid     :", rec["valid_sample"])
    print("motion    :", rec["motion_stats"])